# TSM Development Snapshot recovery — RECOVER_EXISTING

[Open in Colab](https://colab.research.google.com/github/hh4832/market-research-core/blob/feature/phase4a-tsm-native-research-adapter/notebooks/TSM_Development_Snapshot_Recovery.ipynb)

Use a fresh Colab runtime. Complete **all human operations in the first code cell**:
mount Drive, select one existing failed snapshot, confirm paths and mode, enter its
trusted binding SHA, original plan/producer pins and original personal approval
file. Keep the two snapshots' evidence separate. Later cells have no prompts.

This notebook recovers existing Development-only output. It never plans or runs
conversion and never opens original mixed Parquet. It preserves the failed source,
including FAILED.json. A local renewed receipt is not portable back to Drive.


In [ ]:
# ALL HUMAN OPERATIONS / SETTINGS ARE IN THIS FIRST CODE CELL.
from pathlib import Path
from datetime import datetime, timezone
from google.colab import drive
drive.mount('/content/drive')

CORE_SHA = "21d0936b467fc1ea94ab434e086b212bfb6868f4"  # recovery implementation, separate from original producer
EXECUTION_MODE = "RECOVER_EXISTING"
SOURCE_PARENT = Path('/content/drive/MyDrive/00Quant_Research/data/development_snapshots')
LOCAL_PARENT = Path('/content/tsm_development_recovery')  # stable local disk, never Drive
ARCHIVE_PARENT = Path('/content/drive/MyDrive/00Quant_Research/data/development_recovery_archives')

# Fill each record with THAT snapshot's independently recorded binding SHA,
# original approved plan SHA, full producer commit and original approval JSON path.
# Do not substitute one snapshot's evidence for another. No approval is created here.
RECOVERY_CASES = {
    'first': {'snapshot_id': '__SET_FIRST_SNAPSHOT_ID__',
        'binding_sha256': '__SET_TRUSTED_BINDING_SHA256__',
        'plan_sha256': '__SET_ORIGINAL_APPROVED_PLAN_SHA256__',
        'producer_sha': '__SET_ORIGINAL_PRODUCER_COMMIT__',
        'approval_path': '__SET_FIRST_ORIGINAL_APPROVAL_JSON_PATH__'},
    'second': {'snapshot_id': '__SET_SECOND_SNAPSHOT_ID__',
        'binding_sha256': '__SET_TRUSTED_BINDING_SHA256__',
        'plan_sha256': '__SET_ORIGINAL_APPROVED_PLAN_SHA256__',
        'producer_sha': '__SET_ORIGINAL_PRODUCER_COMMIT__',
        'approval_path': '__SET_SECOND_ORIGINAL_APPROVAL_JSON_PATH__'},
}
SELECTED_CASE = ""  # personally select 'first' or 'second'; recover one per run
CONFIRM_RECOVERY_PATHS_AND_EVIDENCE = False  # set True only after checking settings above

import re
assert EXECUTION_MODE == 'RECOVER_EXISTING', 'STOP: only recovery is permitted'
assert CONFIRM_RECOVERY_PATHS_AND_EVIDENCE, 'STOP: confirm first-cell settings'
assert SELECTED_CASE in RECOVERY_CASES, 'STOP: select one snapshot'
CASE = RECOVERY_CASES[SELECTED_CASE]
assert re.fullmatch(r'TSM_DEV_1M_20161019_20241231_[0-9a-f]{20}', CASE['snapshot_id'])
assert re.fullmatch('[0-9a-f]{64}', CASE['binding_sha256'])
assert re.fullmatch('[0-9a-f]{64}', CASE['plan_sha256'])
assert re.fullmatch('[0-9a-f]{40}', CASE['producer_sha'])
assert re.fullmatch('[0-9a-f]{40}', CORE_SHA)
SOURCE_ROOT = (SOURCE_PARENT / CASE['snapshot_id']).resolve(strict=True)
BINDING_PATH = SOURCE_ROOT / 'binding.json'
APPROVAL_EVIDENCE_PATH = Path(CASE['approval_path']).resolve(strict=True)
assert BINDING_PATH.is_file() and APPROVAL_EVIDENCE_PATH.is_file()
assert SOURCE_ROOT.is_relative_to(Path('/content/drive').resolve())
assert ARCHIVE_PARENT.resolve().is_relative_to(Path('/content/drive').resolve())
assert LOCAL_PARENT.resolve().is_relative_to(Path('/content').resolve())
assert not LOCAL_PARENT.resolve().is_relative_to(Path('/content/drive').resolve()), 'STOP: use local disk'
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
LOCAL_DIRECTORY = LOCAL_PARENT / (CASE['snapshot_id'] + '_' + RUN_ID)
ARCHIVE_DIRECTORY = ARCHIVE_PARENT / (CASE['snapshot_id'] + '_' + RUN_ID)
assert not LOCAL_DIRECTORY.exists() and not ARCHIVE_DIRECTORY.exists()
print('Mode:', EXECUTION_MODE, '| selected:', SELECTED_CASE)
print('Existing Development source:', SOURCE_ROOT)
print('New local staging:', LOCAL_DIRECTORY)
print('New exclusive Drive archive:', ARCHIVE_DIRECTORY)
print('Original plan:', CASE['plan_sha256'], '| original producer:', CASE['producer_sha'])
print('Original approval evidence:', APPROVAL_EVIDENCE_PATH)


In [ ]:
# Automated setup only. No additional human operations or prompts.
import subprocess, sys, importlib, json
CORE_ROOT = Path('/content/market-research-core-recovery-' + CORE_SHA)
if not CORE_ROOT.exists():
    subprocess.run(['git', 'clone', 'https://github.com/hh4832/market-research-core.git', str(CORE_ROOT)], check=True)
    subprocess.run(['git', '-C', str(CORE_ROOT), 'checkout', '--detach', CORE_SHA], check=True)
assert subprocess.check_output(['git', '-C', str(CORE_ROOT), 'rev-parse', 'HEAD'], text=True).strip() == CORE_SHA
assert not subprocess.check_output(['git', '-C', str(CORE_ROOT), 'status', '--porcelain'], text=True).strip()
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(CORE_ROOT)], check=True)
sys.path.insert(0, str(CORE_ROOT / 'src'))
importlib.invalidate_caches()
from market_research_core.tsm_native import core_code_identity
from market_research_core.tsm_development import file_sha
from market_research_core.tsm_recovery import recover_development_snapshot
identity = core_code_identity()
assert identity['core_git_sha'] == CORE_SHA and identity['core_tree_dirty'] is False, identity


In [ ]:
# Read only the selected Development binding and its own original approval evidence.
assert file_sha(BINDING_PATH) == CASE['binding_sha256'], 'STOP: trusted binding SHA mismatch'
bundle = json.loads(BINDING_PATH.read_text())
assert bundle['snapshot_pin']['snapshot_id'] == CASE['snapshot_id'], 'STOP: wrong snapshot selected'
assert bundle['snapshot_pin']['producer_sha'] == CASE['producer_sha'], 'STOP: wrong producer'
# Relocation changes the physical root only; manifest, lineage and old receipt pins stay intact.
bundle['snapshot_root'] = str(SOURCE_ROOT)
original_approval = json.loads(APPROVAL_EVIDENCE_PATH.read_text())
assert original_approval.get('plan_sha256') == CASE['plan_sha256'], 'STOP: approval belongs to another plan'
print('Binding SHA: PASS; original evidence will be checked against this snapshot lineage.')


In [ ]:
result = recover_development_snapshot(bundle, LOCAL_DIRECTORY, ARCHIVE_DIRECTORY,
    expected_plan_sha256=CASE['plan_sha256'], expected_producer_sha=CASE['producer_sha'],
    conversion_approval=original_approval, recovery_sha=CORE_SHA)
assert result['local_preflight']['status'] == 'PASS'
marker = json.loads((ARCHIVE_DIRECTORY / 'RECOVERY_VERIFIED.json').read_text())
assert marker['status'] == 'PASS'
assert file_sha(ARCHIVE_DIRECTORY / 'recovery_provenance.json') == marker['recovery_provenance_sha256']
print('Local full preflight: PASS')
print('Archive byte read-back: PASS')
print('Local binding:', LOCAL_DIRECTORY / 'recovery_binding.local.json')
print('Original FAILED.json preserved in source and archived copy.')
print('Archive:', ARCHIVE_DIRECTORY)
print('STOP: no conversion, research authorization or backtest performed.')


## Stop and preserve evidence

Require both local preflight PASS and the new archive's RECOVERY_VERIFIED.json.
Inspect recovery_provenance.json for original failure/stat, plan/producer/evidence,
recovery commit, all file hashes and the local receipt. It records this recovery,
not new conversion approval. The original FAILED.json remains unchanged.

The archived recovery_binding.local.json is evidence of a local validation, **not a
Drive-valid receipt**. On a later runtime/mount, select the archived TSM_DEV directory
as source and stage/renew again using its original pins and approval. Do not patch
receipt metadata manually. To recover the second snapshot, edit only the first
code cell to select its own settings and rerun with new exclusive directories.

Any failed hash, lineage, approval or daily boundary check stops this notebook.
There is no fallback to original conversion; necessity and fresh personal
conversion authorization must be handled separately before any mixed-source read.
